# 04 — Satellite Overpass Prototype

**Story**: S01-04 | **Spec**: `openspec/capabilities/satellite/spec.md`

Two data sources for overpass prediction over the Warnemünde / German Baltic AOI:

1. **CDSE OData** — past Sentinel-2 and Sentinel-3 acquisitions (confirms orbit patterns, returns actual product metadata)
2. **CelesTrak TLEs + skyfield** — forward overpass prediction for all 5 platforms (Suomi-NPP VIIRS, NOAA-20 VIIRS, Landsat 8, Landsat 9, Sentinel-2 A/B via TLE)

> NASA CMR returns no results for recent dates due to processing lag — TLE propagation is more reliable for future planning.

**Platforms**: Sentinel-2 A/B, Sentinel-3 A/B, Landsat 8/9, VIIRS (Suomi-NPP, NOAA-20)  
**AOI**: `[9.5, 53.5, 15.0, 56.0]` (lon_min, lat_min, lon_max, lat_max)

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone, timedelta

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from shapely.geometry import box, LineString, Point
from skyfield.api import load, EarthSatellite, wgs84
import suncalc
from dotenv import load_dotenv

load_dotenv(Path('../.env'))
CDSE_EMAIL    = os.getenv('CDSE_EMAIL')
CDSE_PASSWORD = os.getenv('CDSE_PASSWORD')
assert CDSE_EMAIL and CDSE_PASSWORD, 'CDSE credentials missing from .env'

AOI  = dict(lon_min=9.5, lat_min=53.5, lon_max=15.0, lat_max=56.0)
LOC  = dict(lat=54.18, lon=12.08, name='Warnemünde')
AOI_BOX = box(AOI['lon_min'], AOI['lat_min'], AOI['lon_max'], AOI['lat_max'])

now_utc   = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
ts        = load.timescale()

print(f'Now (UTC): {now_utc.strftime("%Y-%m-%d %H:%M")}')
print(f'AOI: {AOI}')

## 1. CDSE Auth — OAuth2 Bearer Token

In [ ]:
def get_cdse_token(email: str, password: str) -> str:
    """Fetch a CDSE OAuth2 bearer token."""
    resp = requests.post(
        'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
        data={
            'client_id': 'cdse-public',
            'username': email,
            'password': password,
            'grant_type': 'password',
        },
        timeout=15,
    )
    resp.raise_for_status()
    return resp.json()['access_token']

TOKEN = get_cdse_token(CDSE_EMAIL, CDSE_PASSWORD)
HEADERS = {'Authorization': f'Bearer {TOKEN}'}
print(f'CDSE token OK (length: {len(TOKEN)} chars)')

## 2. CDSE OData — Sentinel-2 Acquisitions (Past 10 Days)

Query L2A products intersecting the Baltic AOI. Returns actual acquired products with cloud cover, footprint, orbit number. (SCENARIO-SAT-S2-001)

In [ ]:
ODATA_URL = 'https://catalogue.dataspace.copernicus.eu/odata/v1/Products'

def build_wkt_bbox(lon_min, lat_min, lon_max, lat_max) -> str:
    return (f'POLYGON(({lon_min} {lat_min},{lon_max} {lat_min},'
            f'{lon_max} {lat_max},{lon_min} {lat_max},{lon_min} {lat_min}))')

def query_cdse(token_headers, collection, product_type, start_dt, end_dt,
               bbox=AOI, max_results=100) -> list:
    """Query CDSE OData for products of a given collection/type in a bbox and time range."""
    wkt = build_wkt_bbox(bbox['lon_min'], bbox['lat_min'],
                         bbox['lon_max'], bbox['lat_max'])
    start_s = start_dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')
    end_s   = end_dt.strftime('%Y-%m-%dT%H:%M:%S.000Z')

    filter_parts = [
        f"Collection/Name eq '{collection}'",
        f"OData.CSC.Intersects(area=geography'SRID=4326;{wkt}')",
        f"ContentDate/Start gt {start_s}",
        f"ContentDate/Start lt {end_s}",
    ]
    if product_type:
        filter_parts.append(
            f"Attributes/OData.CSC.StringAttribute/any("
            f"att:att/Name eq 'productType' and "
            f"att/OData.CSC.StringAttribute/Value eq '{product_type}')"
        )

    params = {
        '$filter':  ' and '.join(filter_parts),
        '$orderby': 'ContentDate/Start asc',
        '$top':     max_results,
        '$expand':  'Attributes',
    }
    resp = requests.get(ODATA_URL, params=params,
                        headers=token_headers, timeout=30)
    resp.raise_for_status()
    return resp.json().get('value', [])

def parse_cdse_products(items: list, platform: str) -> list:
    """Extract key fields from CDSE OData product list into a list of dicts."""
    out = []
    for p in items:
        attrs = {a['Name']: a.get('Value', '') for a in p.get('Attributes', [])}
        start = pd.to_datetime(p['ContentDate']['Start'])
        out.append(dict(
            platform    = platform,
            start_utc   = start,
            name        = p['Name'],
            cloud_cover = float(attrs.get('cloudCover', np.nan)),
            orbit       = attrs.get('orbitNumber', ''),
            footprint   = p.get('GeoFootprint', {}),
        ))
    return out

# Query last 10 days of Sentinel-2 L2A
start_q = now_utc - timedelta(days=10)
s2_raw  = query_cdse(HEADERS, 'SENTINEL-2', 'S2MSI2A', start_q, now_utc)
s2_passes = parse_cdse_products(s2_raw, 'Sentinel-2')

print(f'Sentinel-2 L2A acquisitions (last 10 days): {len(s2_passes)}')
if s2_passes:
    df_s2 = pd.DataFrame(s2_passes)
    print(df_s2[['platform','start_utc','cloud_cover','orbit','name']].to_string(index=False))

In [ ]:
# SCENARIO-SAT-S2-001 validation
assert len(s2_passes) >= 1, 'Expected ≥1 Sentinel-2 acquisition over 10-day window'
p0 = s2_passes[0]
assert 'start_utc'   in p0, 'Missing start_utc'
assert 'footprint'   in p0, 'Missing footprint geometry'
assert 'cloud_cover' in p0, 'Missing cloud_cover'
assert p0['footprint'].get('type') in ('Polygon', 'MultiPolygon'), \
    f"Unexpected footprint type: {p0['footprint'].get('type')}"
print(f'✓ SCENARIO-SAT-S2-001 PASS')
print(f'  First acquisition: {p0["start_utc"]} | cloud: {p0["cloud_cover"]:.1f}% | orbit: {p0["orbit"]}')
print(f'  Footprint type: {p0["footprint"]["type"]}')

## 3. CDSE OData — Sentinel-3 Acquisitions (Past 10 Days)

In [ ]:
# Sentinel-3 OLCI Full Resolution (ocean colour / coastal water)
s3_raw    = query_cdse(HEADERS, 'SENTINEL-3', 'OL_1_EFR___', start_q, now_utc)
s3_passes = parse_cdse_products(s3_raw, 'Sentinel-3')

print(f'Sentinel-3 OLCI EFR acquisitions (last 10 days): {len(s3_passes)}')
if s3_passes:
    df_s3 = pd.DataFrame(s3_passes)
    print(df_s3[['platform','start_utc','orbit']].head(8).to_string(index=False))

## 4. CelesTrak TLE Fetch — All 5 Platforms

New CelesTrak GP endpoint: `https://celestrak.org/NORAD/elements/gp.php?CATNR=<ID>&FORMAT=tle`

In [ ]:
CELESTRAK_GP = 'https://celestrak.org/NORAD/elements/gp.php'

SATELLITE_CATALOG = {
    'Suomi-NPP (VIIRS)':  37849,
    'NOAA-20 (VIIRS)':    43013,
    'Landsat 8':          39084,
    'Landsat 9':          49260,
    'Sentinel-2A':        40697,
    'Sentinel-2B':        42063,
}

def fetch_tle(norad_id: int) -> tuple[str, str, str]:
    """Fetch TLE (name, line1, line2) for a NORAD catalog ID from CelesTrak GP endpoint."""
    resp = requests.get(CELESTRAK_GP,
                        params={'CATNR': norad_id, 'FORMAT': 'tle'},
                        timeout=15)
    resp.raise_for_status()
    lines = [l.strip() for l in resp.text.strip().splitlines() if l.strip()]
    assert len(lines) == 3, f'Expected 3 TLE lines for NORAD {norad_id}, got {len(lines)}'
    return lines[0], lines[1], lines[2]

satellites = {}
for name, norad_id in SATELLITE_CATALOG.items():
    sat_name, line1, line2 = fetch_tle(norad_id)
    satellites[name] = EarthSatellite(line1, line2, sat_name, ts)
    epoch_dt = satellites[name].epoch.utc_datetime()
    age_days = (now_utc - epoch_dt).total_seconds() / 86400
    status = '✓' if age_days < 7 else '⚠'
    print(f'{status} {name:25s} | NORAD {norad_id} | TLE epoch: {epoch_dt.strftime("%Y-%m-%d")} ({age_days:.1f}d ago)')

## 5. Skyfield Overpass Prediction — 7-day Forward Look

For each satellite, find passes over the AOI for the next 7 days. Strategy:
- Use `find_events()` with the AOI center (Warnemünde) as the observer point
- For each pass, sample the ground track every 30 s and check intersection with AOI polygon using shapely
- Only keep passes where the ground track actually crosses the AOI (filters out passes that are visible but not overhead)

In [ ]:
AOI_POLY = box(AOI['lon_min'], AOI['lat_min'], AOI['lon_max'], AOI['lat_max'])
OBSERVER = wgs84.latlon(LOC['lat'], LOC['lon'])

t_start = ts.from_datetime(now_utc)
t_end   = ts.from_datetime(now_utc + timedelta(days=7))

def predict_passes(satellite, t0, t1, aoi_poly,
                   alt_deg_min=0.0, track_step_s=30) -> list:
    """
    Find passes of `satellite` over `OBSERVER` between t0 and t1.
    Returns list of dicts with AOS, LOS, peak elevation, and ground track.
    Only passes whose ground track intersects aoi_poly are returned.
    """
    try:
        times, events = satellite.find_events(
            OBSERVER, t0, t1, altitude_degrees=alt_deg_min
        )
    except Exception:
        return []

    passes = []
    i = 0
    while i < len(events):
        if events[i] == 0:  # AOS
            aos_t  = times[i]
            los_t  = None
            peak_t = None
            for j in range(i + 1, min(i + 10, len(events))):
                if events[j] == 1:
                    peak_t = times[j]
                if events[j] == 2:
                    los_t = times[j]
                    break
            if los_t is None:
                i += 1
                continue

            # Sample ground track
            n_steps     = max(2, int((los_t - aos_t) * 86400 / track_step_s))
            track_times = ts.linspace(aos_t, los_t, n_steps)
            subpts      = wgs84.subpoint_of(satellite.at(track_times))
            track_lons  = subpts.longitude.degrees
            track_lats  = subpts.latitude.degrees
            track_line  = LineString(zip(track_lons, track_lats))

            # Only keep passes whose ground track crosses the AOI
            if not track_line.intersects(aoi_poly):
                i += 1
                continue

            # Peak elevation (peak_t is a skyfield Time scalar — use `is not None`)
            if peak_t is not None:
                alt_az    = (satellite - OBSERVER).at(peak_t).altaz()
                peak_elev = float(alt_az[0].degrees)
            else:
                peak_elev = np.nan

            passes.append(dict(
                aos_utc    = aos_t.utc_datetime(),
                los_utc    = los_t.utc_datetime(),
                peak_elev  = peak_elev,
                track_lons = track_lons.tolist(),
                track_lats = track_lats.tolist(),
            ))
            i = j + 1
        else:
            i += 1
    return passes


all_passes = []
for sat_name, sat in satellites.items():
    passes = predict_passes(sat, t_start, t_end, AOI_POLY)
    print(f'{sat_name:25s}: {len(passes):2d} passes over AOI in next 7 days')
    for p in passes:
        p['platform'] = sat_name
    all_passes.extend(passes)

print(f'\nTotal: {len(all_passes)} passes across all platforms')

## 6. Annotate Passes — Cloud Cover + Solar Elevation

In [ ]:
# Fetch Open-Meteo cloud cover for the 7-day window
om_resp = requests.get('https://api.open-meteo.com/v1/forecast', params={
    'latitude': LOC['lat'], 'longitude': LOC['lon'],
    'hourly': 'cloudcover',
    'forecast_days': 7,
    'timezone': 'UTC',
}, timeout=10)
om_resp.raise_for_status()
om = om_resp.json()['hourly']
cc_index = pd.DatetimeIndex(pd.to_datetime(om['time']))
cc_series = pd.Series(om['cloudcover'], index=cc_index)
print(f'Cloud cover series: {len(cc_series)} hourly steps')

def cloud_at(dt: datetime) -> float:
    """Nearest-in-time cloud cover (%) at AOI center."""
    t = pd.Timestamp(dt).tz_localize(None)
    idx = cc_series.index.get_indexer([t], method='nearest')[0]
    return float(cc_series.iloc[idx])

def solar_elev_at(lat: float, lon: float, dt: datetime) -> float:
    """Solar elevation (degrees) at a given location and UTC datetime."""
    pos = suncalc.get_position(dt, lon, lat)
    return float(np.degrees(pos['altitude']))

# Annotate each pass with midpoint time cloud cover and solar elevation
for p in all_passes:
    mid = p['aos_utc'] + (p['los_utc'] - p['aos_utc']) / 2
    p['cloud_pct']  = cloud_at(mid)
    p['solar_elev'] = solar_elev_at(LOC['lat'], LOC['lon'], mid)
    p['mid_utc']    = mid

df = pd.DataFrame(all_passes).sort_values('aos_utc').reset_index(drop=True)
print(f'\nOverpass table ({len(df)} passes):')
print(df[['platform','aos_utc','los_utc','peak_elev','cloud_pct','solar_elev']]
      .to_string(index=False, float_format='%.1f'))

## 7. SCENARIO-SAT-VIIRS-001 Validation

In [ ]:
viirs_passes = df[df['platform'].str.contains('VIIRS|Suomi|NOAA-20', case=False)]
assert len(viirs_passes) >= 1, 'Expected ≥1 VIIRS overpass in 7-day window'
vp = viirs_passes.iloc[0]
duration_min = (vp['los_utc'] - vp['aos_utc']).total_seconds() / 60
print(f'✓ SCENARIO-SAT-VIIRS-001 PASS')
print(f'  Platform: {vp["platform"]}')
print(f'  AOS:  {vp["aos_utc"].strftime("%Y-%m-%d %H:%M")} UTC')
print(f'  LOS:  {vp["los_utc"].strftime("%Y-%m-%d %H:%M")} UTC')
print(f'  Duration: {duration_min:.1f} min')
print(f'  Ground track length: {len(vp["track_lons"])} points')
print(f'  Cloud cover at pass: {vp["cloud_pct"]:.0f}%')
print(f'  Solar elevation:     {vp["solar_elev"]:.1f}°')

## 8. Timeline Visualization

In [ ]:
PLATFORM_COLORS = {
    'Suomi-NPP (VIIRS)': '#E91E63',
    'NOAA-20 (VIIRS)':   '#9C27B0',
    'Landsat 8':         '#FF5722',
    'Landsat 9':         '#FF9800',
    'Sentinel-2A':       '#2196F3',
    'Sentinel-2B':       '#03A9F4',
}

platforms = list(PLATFORM_COLORS.keys())
y_pos     = {p: i for i, p in enumerate(platforms)}

fig, (ax_tl, ax_cc) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})

# Timeline: horizontal bars, one per pass
for _, row in df.iterrows():
    y   = y_pos.get(row['platform'], len(platforms))
    col = PLATFORM_COLORS.get(row['platform'], 'gray')
    ax_tl.barh(y, left=row['aos_utc'],
               width=row['los_utc'] - row['aos_utc'],
               color=col, alpha=0.8, height=0.6)
    # Dim passes that are night-time
    if row['solar_elev'] < 0:
        ax_tl.barh(y, left=row['aos_utc'],
                   width=row['los_utc'] - row['aos_utc'],
                   color='black', alpha=0.3, height=0.6)

ax_tl.set_yticks(range(len(platforms)))
ax_tl.set_yticklabels(platforms)
ax_tl.axvline(now_utc.replace(tzinfo=None), color='red', ls='--', lw=1.5, label='Now')
ax_tl.set_title('Satellite Overpasses over Baltic AOI — 7-day Forward Look\n'
                 '(dark overlay = night pass; bars are AOS–LOS duration)')
ax_tl.legend(fontsize=9)
ax_tl.grid(axis='x', alpha=0.3)

# Cloud cover ribbon
now_naive = pd.Timestamp(now_utc).tz_localize(None)
cc_future = cc_series[cc_series.index >= now_naive]
ax_cc.fill_between(cc_future.index, cc_future.values, alpha=0.5, color='gray')
ax_cc.set_ylim(0, 100)
ax_cc.set_ylabel('Cloud\ncover (%)')
ax_cc.axhline(20, color='green', ls=':', lw=1)
ax_cc.grid(axis='x', alpha=0.3)
ax_cc.xaxis.set_major_formatter(mdates.DateFormatter('%a %d %b\n%H:%M'))
ax_cc.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))

plt.tight_layout()
plt.show()

## 9. Best Optical Windows

Flag passes with solar elevation > 20° and cloud cover < 30% — the optimal Go windows for optical surveys.

In [ ]:
good = df[(df['solar_elev'] > 20) & (df['cloud_pct'] < 30)].copy()
print(f'Optimal passes (solar > 20°, cloud < 30%): {len(good)}')
if not good.empty:
    print(good[['platform','aos_utc','solar_elev','cloud_pct']]
          .to_string(index=False, float_format='%.1f'))
else:
    print('  None in current forecast window — check back when cloud cover improves')

## 10. SCENARIO-SAT-FAIL-001 — Graceful Error Handling

In [ ]:
def query_cdse_safe(headers, collection, product_type, start_dt, end_dt) -> dict:
    try:
        items = query_cdse(headers, collection, product_type, start_dt, end_dt)
        return {'status': 'ok', 'data': items}
    except Exception as e:
        return {'status': 'error', 'layer': 'satellite', 'message': str(e)}

# Simulate failure with a bad token
bad_headers = {'Authorization': 'Bearer invalid_token_xyz'}
result = query_cdse_safe(bad_headers, 'SENTINEL-2', 'S2MSI2A', start_q, now_utc)
assert result['status'] == 'error'
print('✓ SCENARIO-SAT-FAIL-001 PASS')
print(f'  Error message: {result["message"][:100]}')

## 11. Summary of Findings

In [ ]:
print('=== FINDINGS FOR ops/data-sources.md ===')
print()
print('CDSE OData (Sentinel-2/3 past acquisitions):')
print('  Endpoint: https://catalogue.dataspace.copernicus.eu/odata/v1/Products')
print('  Auth: OAuth2 Bearer token from identity.dataspace.copernicus.eu')
print('  Filter: Collection/Name eq + OData.CSC.Intersects + ContentDate/Start range')
print('  Product type filter: Attributes/OData.CSC.StringAttribute/any(...)')
print('  Key attributes: cloudCover (float), platformShortName, orbitNumber')
print('  GeoFootprint: GeoJSON Polygon in product dict')
print()
print('CelesTrak GP endpoint (new, post-2022):')
print('  URL: https://celestrak.org/NORAD/elements/gp.php?CATNR=<ID>&FORMAT=tle')
print('  Old URL (deprecated 2020): celestrak.org/satcat/tle.php — DO NOT USE')
print('  TLE freshness: flag passes if TLE epoch > 7 days old')
print()
print('NASA CMR (Landsat):')
print('  Returns 0 results for recent dates (processing lag ~1-3 days)')
print('  TLE-based prediction is more reliable for FUTURE overpass planning')
print('  CMR still useful for querying PAST acquisitions for cloud cover stats')
print()
print('skyfield overpass method:')
print('  find_events(observer, t0, t1, altitude_degrees=0)')
print('  events: 0=AOS, 1=transit, 2=LOS')
print('  Ground track sampled every 30 s; AOI intersection via shapely')
print(f'  7-day results: {len(df)} passes total across {len(df["platform"].unique())} platforms')
print()
print('Phase 2 design notes:')
print('  - Refresh TLEs daily (cache in data/external/tles/)')
print('  - Token expiry: CDSE tokens last ~1 h; refresh before each batch query')
print('  - Combine CDSE historical + TLE forward for full planning window')
print('  - Sentinel-2A/B NORAD IDs: 40697 / 42063; Sentinel-3A/B: 41335 / 43437')